# Retrain Catboost for convergence

In [ ]:
# Retraining catboost
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from scipy.fft import fft, fftfreq

# Files
DATA_PATH = Path("swat_post_processed.csv")
MODEL_DIR = Path("saved_models")

# Load processed dataset
processed_clean = pd.read_csv(
    DATA_PATH,
    parse_dates=["Timestamp"],
    index_col="Timestamp"
).sort_index()

target = "LIT101"

# Reproduce the original time boundaries
WARMUP_END = pd.Timestamp("2015-12-23 15:00:00")

first_attack = (
    processed_clean.index[
        processed_clean["Normal/Attack"].eq("Attack")
    ].min().floor("min")
)

normal_minutes = pd.date_range(
    max(WARMUP_END, processed_clean.index.min().ceil("min")),
    first_attack - pd.Timedelta(minutes=1),
    freq="min"
)

development_size = int(len(normal_minutes) * 0.8)
TEST_START = normal_minutes[development_size]
VAL_START = normal_minutes[int(development_size * 0.8)]

normal_1min = (
    processed_clean.loc[
        processed_clean.index < first_attack
    ].asfreq("min")
)

assert normal_1min.index.equals(normal_minutes)
assert normal_1min["LIT101"].notna().all()
assert normal_1min["sample_count"].eq(60).all()

ml_df = normal_1min[[target]].copy()

# Reproduce FFT cycle selection from fitting data
s = ml_df.loc[ml_df.index < VAL_START, target]

frequencies = fftfreq(len(s), d=1)
magnitudes = np.abs(fft((s - s.mean()).to_numpy()))

band = (
    (frequencies > 1 / 360) &
    (frequencies < 1 / 5)
)

CYCLE = int(round(
    1 / frequencies[band][np.argmax(magnitudes[band])]
))

print("Detected cycle:", CYCLE)

# Same feature engineering as original notebook
def create_all_features(data, cycle=CYCLE):
    df = data.copy()

    df["hour"] = df.index.hour
    df["sin_hour"] = np.sin(2 * np.pi * df.index.hour / 24)
    df["cos_hour"] = np.cos(2 * np.pi * df.index.hour / 24)

    df["trend"] = (
        (df.index - ml_df.index[0]) /
        pd.Timedelta(minutes=1)
    ).to_numpy()

    df["trend_squared"] = df["trend"] ** 2

    for lag in sorted({1, 2, 3, 6, 12, 24, cycle}):
        df[f"lag_{lag}"] = df[target].shift(lag)

    for window in [3, 6, 12]:
        previous = df[target].shift(1)

        df[f"rolling_mean_{window}"] = (
            previous.rolling(window).mean()
        )
        df[f"rolling_std_{window}"] = (
            previous.rolling(window).std()
        )

    df[f"ratio_lag1_lag{cycle}"] = (
        df["lag_1"] /
        df[f"lag_{cycle}"].replace(0, np.nan)
    )

    feature_cols = df.columns.drop(target)
    df[feature_cols] = df[feature_cols].replace(
        [np.inf, -np.inf], np.nan
    )

    return df.dropna(subset=feature_cols)


features = create_all_features(ml_df)

train_feat = features.loc[features.index < TEST_START]
test_feat = features.loc[features.index >= TEST_START]

X_train_full = train_feat.drop(columns=target)
X_test_full = test_feat.drop(columns=target)

y_train = train_feat[target]
y_test_final = test_feat[target]

split = int((X_train_full.index < VAL_START).sum())

X_fit_full = X_train_full.iloc[:split]
X_val_full = X_train_full.iloc[split:]

y_train_final = y_train.iloc[:split]
y_val_final = y_train.iloc[split:]

# Load EXACT feature selection from saved notebook
selected_features = joblib.load(
    MODEL_DIR / "ml_features.pkl"
)

X_train_final = X_fit_full[selected_features]
X_val_final = X_val_full[selected_features]
X_test_final = X_test_full[selected_features]

assert X_test_final.index.equals(
    ml_df.loc[ml_df.index >= TEST_START].index
)

print("\nDataset successfully reconstructed")
print("Training:", X_train_final.shape)
print("Validation:", X_val_final.shape)
print("Test:", X_test_final.shape)
print("Features:", selected_features)
print("Training ends:", X_train_final.index[-1])
print("Validation starts:", X_val_final.index[0])
print("Test starts:", X_test_final.index[0])


In [ ]:

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error

cat_old = CatBoostRegressor()
cat_old.load_model(str(MODEL_DIR / "catboost_tuned.cbm"))

xgb_old = XGBRegressor()
xgb_old.load_model(str(MODEL_DIR / "xgboost_tuned.json"))

# Verify features before continuing
assert cat_old.feature_names_ == selected_features
assert xgb_old.get_booster().feature_names == selected_features

print("CatBoost trees:", cat_old.tree_count_)
print("XGBoost trees:", xgb_old.get_booster().num_boosted_rounds())

print("CatBoost validation RMSE:",
      root_mean_squared_error(
          y_val_final, cat_old.predict(X_val_final)
      ))

print("XGBoost validation RMSE:",
      root_mean_squared_error(
          y_val_final, xgb_old.predict(X_val_final)
      ))


In [ ]:

cat_extended = CatBoostRegressor(
    iterations=5000,  # Additional iterations
    depth=cat_old.get_param("depth"),
    learning_rate=cat_old.get_param("learning_rate"),
    bootstrap_type=cat_old.get_param("bootstrap_type"),
    subsample=cat_old.get_param("subsample"),
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42
)

cat_extended.fit(
    X_train_final,
    y_train_final,
    init_model=cat_old,
    eval_set=(X_val_final, y_val_final),
    early_stopping_rounds=200,
    verbose=100
)

print("Total trees:", cat_extended.tree_count_)
print("Validation RMSE:",
      root_mean_squared_error(
          y_val_final,
          cat_extended.predict(X_val_final)
      ))

cat_extended.save_model(
    str(MODEL_DIR / "catboost_extended.cbm")
)


In [ ]:

results = pd.DataFrame({
    "Model": [
        "CatBoost Original",
        "CatBoost Extended",
        "XGBoost Original"    ],
    "Validation RMSE": [
        root_mean_squared_error(
            y_val_final, m.predict(X_val_final)
        )
        for m in [
            cat_old, cat_extended,
            xgb_old
        ]
    ]
})

display(results.sort_values("Validation RMSE"))


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

models = {
    "CatBoost Original": cat_old,
    "CatBoost Extended": cat_extended,
    "XGBoost Original": xgb_old
}

test_results = []
predictions = pd.DataFrame(
    {"Actual": y_test_final},
    index=y_test_final.index
)

for name, model in models.items():
    pred = model.predict(X_test_final)
    actual = y_test_final.to_numpy()

    mae = mean_absolute_error(actual, pred)
    mse = mean_squared_error(actual, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(actual, pred)

    nonzero = np.abs(actual) > 1e-8
    mape = np.mean(
        np.abs((actual[nonzero] - pred[nonzero]) / actual[nonzero])
    ) * 100

    test_results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE (%)": mape,
        "R2": r2
    })

    predictions[name] = pred

test_metrics = pd.DataFrame(test_results)

display(test_metrics.sort_values("RMSE"))
display(predictions.head(20))

In [ ]:
from catboost import CatBoostRegressor

cat_fresh = CatBoostRegressor(
    iterations=15000,
    depth=6,
    learning_rate=0.03,
    subsample=1.0,
    bootstrap_type="Bernoulli",
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42
)

cat_fresh.fit(
    X_train_final,
    y_train_final,
    eval_set=(X_val_final, y_val_final),
    early_stopping_rounds=200,
    use_best_model=True,
    verbose=500
)

print("Best iteration:", cat_fresh.best_iteration_)
print("Best validation RMSE:", cat_fresh.best_score_["validation"]["RMSE"])

cat_fresh.save_model("catboost_fresh.cbm")

In [ ]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd

comparison = pd.DataFrame([
    {
        "Model": name,
        "Trees": (
            model.tree_count_
            if "CatBoost" in name
            else model.best_iteration + 1
            if hasattr(model, "best_iteration")
            else model.get_booster().num_boosted_rounds()
        ),
        "Validation RMSE": root_mean_squared_error(
            y_val_final,
            model.predict(X_val_final)
        )
    }
    for name, model in {
        "Original CatBoost": cat_old,
        "Fresh CatBoost": cat_fresh,
        "Original XGBoost": xgb_old,
    }.items()
])

display(comparison.sort_values("Validation RMSE"))

In [ ]:
cat_test_pred = cat_fresh.predict(X_test_final)
cat_test_rmse = np.sqrt(mean_squared_error(y_test_final, cat_test_pred))
print(f"CatBoost Test RMSE: {cat_test_rmse:.4f}")



## What we can use:
The initial CatBoost model reached the maximum allowed 5,000 iterations without triggering early stopping.
so i checked to see if we can improve predictive performance, i retrained the model using the previously selected hyperparameters with a maximum of 15,000 iterations. Early stopping selected iteration 8,578. Test RMSE improved from 3.2135 to 3.1939, representing a 0.61% reduction. These results suggest that the original model was already close to its best achievable performance under the selected configuration.